# HighPitchCT — Stage 0 على Kaggle (النسخة الذكية v0.3.0)

1. فعّل **Internet = On** من إعدادات الدفتر.
2. أضف سرًا في **Add-ons → Secrets** باسم `GITHUB_TOKEN` يحتوي توكن GitHub الخاص بك.
3. يكفي `Accelerator = None` في Stage 0 لتوفير حصة الـ GPU، أو اختر `GPU T4 x2` للتحقق من قراءة البطاقتين في `hpct env`.
4. يكتشف الكود بيئة Kaggle تلقائيًا ويفرض سقف قرص صارمًا **<= 18.0 GiB** داخل `/kaggle/working/highpitchct-data` مع وضع ملفات `.zip` المؤقتة في `/tmp/highpitchct-incoming`.


In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = Path('/kaggle/working/HighPitchCT-Pipeline')
if not repo.exists():
    rc = os.system(f'git clone https://{token}@github.com/BrahimSoufghalem/HighPitchCT-Pipeline.git {repo}')
    assert rc == 0, 'فشل استنساخ المستودع؛ تحقق من Internet ومن GITHUB_TOKEN'

%cd /kaggle/working/HighPitchCT-Pipeline
!python -m pip install -q -e .
!python -m unittest discover -s tests -v


In [ ]:
!hpct env


## تنزيل وفهرسة المريض التجريبي الجاهز `C012`
ملف `configs/pilot_c012.csv` مضمّن وجاهز بـ UIDs الحقيقية للمريض `C012` (الإسقاطات كاملة الجرعة + الصور المنشورة للتحقق البصري فقط — نحو 1.45 GB).


In [ ]:
!hpct validate-manifest --series-file configs/pilot_c012.csv
!hpct download --series-file configs/pilot_c012.csv --workers 2 --retries 5
!hpct index


In [ ]:
import pandas as pd
inv = pd.read_csv('/kaggle/working/highpitchct-data/manifests/series_inventory.csv')
inv[[
    'series_description',
    'dicom_kind',
    'file_count',
    'total_mb',
    'ffs_mode',
    'axial_span_mm',
    'estimated_table_feed_mm_per_rot',
    'estimated_normalized_pitch',
    'pitch_fit_rmse_mm',
    'validation_status',
]]
